# Ising Model: Magnetization & Susceptibility Analysis

This notebook performs a parameter sweep over $\beta$ (inverse temperature) for the 2D Ising model. 
It measures the magnetization evolution and calculates the magnetic susceptibility to identify the phase transition.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from spin_engine.models.ising import IsingSystem
from spin_engine.dynamics.metropolis import MetropolisDynamics
from spin_engine.interactions.standard import PeriodicNearestNeighborInteraction
from spin_engine.measurements.tracker import Tracker
from spin_engine.measurements.scalars import Magnetization

In [ ]:
# --- Configuration ---
L = 16  # Lattice size (LxL)
replicas = 5  # Number of independent replicas per beta
steps = 1000  # Number of MCMC steps
granularity = 1  # Record every N steps

# Beta range (Critical beta for 2D Ising is ~0.44)
# We sweep around this value
betas = np.linspace(0.0, 1.0, 20)

lattice_dim = 2

In [ ]:
# --- Generate Interaction Matrix ---
interaction_gen = PeriodicNearestNeighborInteraction()
interaction_matrix = interaction_gen.generate(D=lattice_dim, L=L)
interaction_matrix = tf.convert_to_tensor(interaction_matrix, dtype=tf.float32)

print(f"Interaction Matrix Shape: {interaction_matrix.shape}")

In [ ]:
# --- Simulation Loop ---
magnetization_histories = []

dynamics = MetropolisDynamics()

print("Starting sweep...")
for beta in betas:
    print(f"Processing beta={beta:.3f}...")
    
    # Initialize System (Cold start or Hot start? Default is random/hot)
    system = IsingSystem(
        lattice_length=L,
        lattice_replicas=replicas,
        interaction_matrix=interaction_matrix,
        lattice_dim=lattice_dim,
        initial_magnetization=0.0 # Random start
    )
    system.initialize_state()
    
    # Setup Tracker
    tracker = Tracker(
        measurements=[Magnetization()],
        sweep_length=steps,
        granularity=granularity
    )
    
    # Run Dynamics
    # Note: Magnetization measurement returns shape (replicas,)
    sweep_results = dynamics.sweep(
        system=system,
        n_steps=steps,
        tracker=tracker,
        beta=beta,
        num_disturb=1 # Flip 1 spin per step (standard Metropolis)
    )
    
    # Collect Magnetization
    # shape: (steps/granularity, replicas)
    mag_history = sweep_results['Magnetization'] 
    magnetization_histories.append(mag_history)

# Stack results
# shape: (n_betas, steps, replicas)
magnetization_evolution = tf.stack(magnetization_histories)

results = {
    'betas': betas,
    'magnetization_evolution': magnetization_evolution
}

print("Sweep complete.")

## Magnetization Evolution

In [ ]:
# User-provided plotting code

# Collect betas and corresponding magnetization arrays
betas_sorted = results['betas']
magnetization_data = abs(results["magnetization_evolution"].numpy())
lattice_length = L # For susceptibility calculation later

n_betas = len(betas_sorted)
steps   = magnetization_data[0].shape[0]
replicas = magnetization_data[0].shape[1]

print(f"Visualizing {n_betas} betas. Data shape: {magnetization_data.shape}")

# Plot evolution for each beta
# Adjust figsize to be reasonable if n_betas is large
fig, axes = plt.subplots(nrows=n_betas, ncols=1, figsize=(10, 2*n_betas), sharex=True)

if n_betas == 1:
    axes = [axes]  # ensure iterable if only 1 beta

for idx, beta in enumerate(betas_sorted):
    ax = axes[idx]
    magnetization_beta = magnetization_data[idx]  # shape (steps, replicas)

    # plot all replicas
    for r in range(replicas):
        ax.plot(range(steps), magnetization_beta[:, r], alpha=0.4, lw=1)

    # plot mean across replicas
    mean_magnetization = magnetization_beta.mean(axis=1)
    ax.plot(range(steps), mean_magnetization, color="black", lw=2.5, label="Mean")

    ax.set_title(f"\u03b2 = {beta:.3f}")
    ax.set_ylabel("Magnetization")
    ax.legend(loc='upper right')

axes[-1].set_xlabel("Measurement step")
plt.tight_layout()
plt.show()

## Final Magnetization & Susceptibility

In [ ]:
# --- Final magnetization (last step) ---
final_magnetization_replicas = np.array([m[-1, :] for m in magnetization_data])  # (n_betas, replicas)

# Compute mean and std across replicas
final_magnetization_mean = final_magnetization_replicas.mean(axis=1)
final_magnetization_std  = final_magnetization_replicas.std(axis=1)

plt.figure(figsize=(24, 6))

# Plot each replica (faded for readability)
for r in range(replicas):
    plt.plot(
        betas_sorted,
        final_magnetization_replicas[:, r],
        lw=1,
        alpha=0.5,
        label="Replica" if r == 0 else None,
    )

# Overlay average across replicas in black with error bars
plt.errorbar(
    betas_sorted,
    final_magnetization_mean,
    yerr=final_magnetization_std,
    color="black",
    elinewidth=0.5,
    capsize=4,
    lw=0.7,
    label="Replica mean \u00b1 std"
)

plt.xlabel(r"$\beta$")
plt.ylabel("Magnetization")
plt.title("Magnetization vs \u03b2 (mean \u00b1 std)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


# --- Susceptibility ---
N = lattice_length ** lattice_dim  # total number of spins

# Variance across replicas
final_magnetization_var = final_magnetization_replicas.var(axis=1)

# Susceptibility (\u03c7 = \u03b2 N Var(M))
susceptibility = np.array(betas_sorted) * N * final_magnetization_var

plt.figure(figsize=(12, 6))
plt.plot(
    betas_sorted,
    susceptibility,
    lw=0.7,
    color="black",
    label=r"Magnetic susceptibility $\chi$",
)

plt.xlabel(r"$\beta$")
plt.ylabel(r"$\chi$")
plt.title(r"Magnetic Susceptibility vs $\beta$")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()